In [ ]:
# ============================================================
# MIDI SONIFICATION — Verkaufsprognose als Musik
# Konzept: Wöchentlicher Arpeggio in A-Moll, 120 BPM
# Daten:   Random Forest Prognose Januar-März 2014
# ============================================================
import pandas as pd
import numpy as np
import joblib
import os
from midiutil import MIDIFile

# --- Pfade ---
DATA_PROC  = "../data/processed/"
DATA_RAW   = "../data/raw/"
MODEL_PATH = "../models/champion_model.pkl"
MIDI_OUT   = "../data/midi/"
os.makedirs(MIDI_OUT, exist_ok=True)

# --- Champion Model & Daten laden ---
model      = joblib.load(MODEL_PATH)
df         = pd.read_csv(DATA_PROC + "timeseries_cleaned.csv",
                         index_col='date', parse_dates=True)
df_oil     = pd.read_csv(DATA_PROC + "oil_cleaned.csv",
                         index_col='date', parse_dates=True)
df_holiday = pd.read_csv(DATA_RAW + "holidays.csv",
                         parse_dates=['date'])

print("✅ Modell und Daten geladen")

# ============================================================
# FEATURE ENGINEERING (identisch mit app.py)
# ============================================================
FEATURE_COLS = [
    'day_of_week', 'day_of_month', 'month', 'week_of_year',
    'is_weekend', 'lag_1', 'lag_7', 'lag_14',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7',
    'rolling_max_7', 'rolling_min_7', 'oil_price', 'is_holiday'
]

def predict_n_days(model, df_feat, start_date, n_days,
                   df_oil, df_holiday):

    national    = df_holiday[
        df_holiday['locale'] == 'National'
    ]['date'].dt.date.tolist()
    local_quito = df_holiday[
        (df_holiday['locale'] == 'Local') &
        (df_holiday['locale_name'] == 'Quito')
    ]['date'].dt.date.tolist()
    alle_feiertage = list(set(national + local_quito))

    # Historische Verkaufswerte als Dictionary
    # → kein DataFrame-Index-Problem möglich
    sales_history = df_feat['unit_sales'].to_dict()

    predictions  = []
    current_date = start_date

    for _ in range(n_days):

        # Lag-Werte direkt aus Dictionary
        lag_1  = float(sales_history.get(
            current_date - pd.Timedelta(days=1),  np.nan))
        lag_7  = float(sales_history.get(
            current_date - pd.Timedelta(days=7),  np.nan))
        lag_14 = float(sales_history.get(
            current_date - pd.Timedelta(days=14), np.nan))

        # Rolling aus letzten N Einträgen des Dictionary
        sorted_dates = sorted(
            [d for d in sales_history.keys() if d < current_date]
        )
        last_7  = [sales_history[d] for d in sorted_dates[-7:]]
        last_14 = [sales_history[d] for d in sorted_dates[-14:]]

        row = {
            'day_of_week':    float(current_date.dayofweek),
            'day_of_month':   float(current_date.day),
            'month':          float(current_date.month),
            'week_of_year':   float(current_date.isocalendar()[1]),
            'is_weekend':     float(current_date.dayofweek >= 5),
            'lag_1':          lag_1,
            'lag_7':          lag_7,
            'lag_14':         lag_14,
            'rolling_mean_7':  float(np.mean(last_7))  if last_7  else np.nan,
            'rolling_mean_14': float(np.mean(last_14)) if last_14 else np.nan,
            'rolling_std_7':   float(np.std(last_7))   if len(last_7) > 1 else 0.0,
            'rolling_max_7':   float(np.max(last_7))   if last_7  else np.nan,
            'rolling_min_7':   float(np.min(last_7))   if last_7  else np.nan,
        }

        # Ölpreis
        if current_date in df_oil.index:
            row['oil_price'] = float(df_oil.loc[current_date, 'dcoilwtico'])
        else:
            row['oil_price'] = float(df_oil['dcoilwtico'].dropna().iloc[-1])

        row['is_holiday'] = float(current_date.date() in alle_feiertage)

        # Vorhersage
        X    = pd.DataFrame([row])[FEATURE_COLS].astype(float)
        pred = max(0.0, float(model.predict(X)[0]))

        # In History speichern
        sales_history[current_date] = pred
        predictions.append({'date': current_date, 'prediction': pred})
        current_date += pd.Timedelta(days=1)

    return pd.DataFrame(predictions).set_index('date')


In [6]:
# ============================================================
# ZELLE 2: Feature Engineering & Prognose ausführen
# ============================================================

def build_features(df, df_oil, df_holiday):
    df_feat = df.copy()
    df_feat['day_of_week']  = df_feat.index.dayofweek
    df_feat['day_of_month'] = df_feat.index.day
    df_feat['month']        = df_feat.index.month
    df_feat['week_of_year'] = df_feat.index.isocalendar().week.astype(int)
    df_feat['is_weekend']   = (df_feat.index.dayofweek >= 5).astype(int)
    df_feat['lag_1']        = df_feat['unit_sales'].shift(1)
    df_feat['lag_7']        = df_feat['unit_sales'].shift(7)
    df_feat['lag_14']       = df_feat['unit_sales'].shift(14)
    df_feat['rolling_mean_7']  = df_feat['unit_sales'].shift(1).rolling(7).mean()
    df_feat['rolling_mean_14'] = df_feat['unit_sales'].shift(1).rolling(14).mean()
    df_feat['rolling_std_7']   = df_feat['unit_sales'].shift(1).rolling(7).std()
    df_feat['rolling_max_7']   = df_feat['unit_sales'].shift(1).rolling(7).max()
    df_feat['rolling_min_7']   = df_feat['unit_sales'].shift(1).rolling(7).min()
    df_feat['oil_price']       = df_oil['dcoilwtico']

    national    = df_holiday[
        df_holiday['locale'] == 'National'
    ]['date'].dt.date.tolist()
    local_quito = df_holiday[
        (df_holiday['locale'] == 'Local') &
        (df_holiday['locale_name'] == 'Quito')
    ]['date'].dt.date.tolist()
    alle_feiertage = list(set(national + local_quito))
    df_feat['is_holiday'] = df_feat.index.date
    df_feat['is_holiday'] = df_feat['is_holiday'].apply(
        lambda x: 1 if x in alle_feiertage else 0
    )
    return df_feat.dropna()

# Feature DataFrame erstellen
df_feat = build_features(df, df_oil, df_holiday)

# Prognose Januar-März 2014 (90 Tage)
forecast = predict_n_days(
    model, df_feat,
    pd.Timestamp('2014-01-01'), 90,
    df_oil, df_holiday
)

print(f"✅ Prognose erstellt: {len(forecast)} Tage")
print(f"   Zeitraum: {forecast.index.min().date()} → {forecast.index.max().date()}")
print(f"   Min:      {forecast['prediction'].min():.0f} Einheiten")
print(f"   Max:      {forecast['prediction'].max():.0f} Einheiten")
print(f"\nVorschau:")
print(forecast.head(7).round(1))

✅ Modell und Daten geladen
✅ Prognose erstellt: 90 Tage
   Zeitraum: 2014-01-01 → 2014-03-31
   Min:      360 Einheiten
   Max:      690 Einheiten

Vorschau:
            prediction
date                  
2014-01-01       414.1
2014-01-02       373.0
2014-01-03       376.4
2014-01-04       629.4
2014-01-05       690.3
2014-01-06       435.6
2014-01-07       381.2


In [7]:
# ============================================================
# ZELLE 3: MIDI Generierung — Arpeggio in A-Moll, 120 BPM
# ============================================================

# --- Tonmapping: Wochentag → MIDI Note ---
# A-Moll Tonleiter: A3, C4, E4, G4, A4, C5, E5
# Mo=0, Di=1, Mi=2, Do=3, Fr=4, Sa=5, So=6
WOCHENTAG_NOTEN = {
    0: 57,   # Mo → A3  (Root)
    1: 60,   # Di → C4  (kleine Terz)
    2: 64,   # Mi → E4  (Quinte)
    3: 67,   # Do → G4  (kleine Septime)
    4: 69,   # Fr → A4  (Oktave)
    5: 72,   # Sa → C5  (laut, Wochenende)
    6: 76,   # So → E5  (lautest, stärkster Tag)
}

# --- Velocity (Lautstärke) aus Verkaufswert ---
# Verkauf 360-690 → Velocity 40-120 (MIDI: 0-127)
def verkauf_zu_velocity(verkauf: float,
                         min_v: float, max_v: float) -> int:
    """Normalisiert Verkaufswert auf MIDI Velocity 40-120."""
    velocity = int(40 + (verkauf - min_v) / (max_v - min_v) * 80)
    return max(40, min(120, velocity))

# --- Notenlänge aus Wochentag ---
# Wochenende = längere Noten (wichtiger Tag)
def notenlaenge(wochentag: int) -> float:
    """Gibt Notenlänge in Beats zurück."""
    if wochentag >= 5:    # Sa/So
        return 1.0        # Ganze Note pro Beat
    else:                 # Werktag
        return 0.75       # Dreiviertel Note

# --- MIDI Datei erstellen ---
TEMPO    = 120   # BPM
TRACK    = 0
CHANNEL  = 0

midi = MIDIFile(1)                    # 1 Track
midi.addTempo(TRACK, 0, TEMPO)

min_verkauf = forecast['prediction'].min()
max_verkauf = forecast['prediction'].max()

beat = 0.0  # aktueller Zeitpunkt in Beats

print("=" * 55)
print("MIDI GENERIERUNG — A-Moll Arpeggio, 120 BPM")
print("=" * 55)

for datum, row in forecast.iterrows():
    wochentag = datum.dayofweek
    verkauf   = float(row['prediction'])
    note      = WOCHENTAG_NOTEN[wochentag]
    velocity  = verkauf_zu_velocity(verkauf, min_verkauf, max_verkauf)
    dauer     = notenlaenge(wochentag)

    # Ausreißer → eine Oktave höher
    if verkauf > forecast['prediction'].quantile(0.95):
        note += 12

    # Feiertag → Pause (keine Note)
    if verkauf == 0:
        beat += dauer
        continue

    midi.addNote(TRACK, CHANNEL, note, beat, dauer, velocity)
    beat += dauer

# --- MIDI Datei speichern ---
midi_datei = MIDI_OUT + "verkaufsprognose_2014.mid"
with open(midi_datei, 'wb') as f:
    midi.writeFile(f)

print(f"✅ MIDI Datei gespeichert: {midi_datei}")
print(f"   Noten:    {len(forecast)} Töne")
print(f"   Dauer:    ~{beat/TEMPO*60:.0f} Sekunden ({beat:.0f} Beats)")
print(f"   Tempo:    {TEMPO} BPM")
print(f"   Tonart:   A-Moll")
print(f"\nNächster Schritt:")
print(f"   1. Öffne: data/midi/verkaufsprognose_2014.mid in Ableton")
print(f"   2. Wähle ein Instrument (z.B. Operator oder Analog)")
print(f"   3. Exportiere als MP3")

MIDI GENERIERUNG — A-Moll Arpeggio, 120 BPM
✅ MIDI Datei gespeichert: ../data/midi/verkaufsprognose_2014.mid
   Noten:    90 Töne
   Dauer:    ~37 Sekunden (74 Beats)
   Tempo:    120 BPM
   Tonart:   A-Moll

Nächster Schritt:
   1. Öffne: data/midi/verkaufsprognose_2014.mid in Ableton
   2. Wähle ein Instrument (z.B. Operator oder Analog)
   3. Exportiere als MP3
